# SnowPredict - Milestone 2: Data Wrangling and Project Redefinition

**Project group #8**
**Group members:** Livia Jonnatan, Jiwon Kim, Jihoon Park

**Project question (original proposal):** Can machine-learning models predict overall skiing conditions at European ski resorts 30 days in advance more accurately than historical averages?

**Data source:** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). Free, no API key.

<div class="alert alert-info" style="color: #4a4a4a; background-color: #fffad2; border-color: #ffedbb; border-width: 1px; border-radius: 3px; padding: 10px;">

**Notebook order follows:**
1. Getting the data
2. Understanding the data
3. Cleaning and preprocessing
4. Project question (revised proposal)

*Conventions:* every fetch is guarded and validated before it is cached, so re-running the notebook never re-downaloads. Nothing is imputed by default.

In [1]:
import warnings

import os, json, time, glob, datetime
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
# Silence scikit-learn's deprecation notices about the `penalty` argument
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*penalty is deprecated.*")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11

# 1. Getting the data

## 1.1 Resorts and representative points

Calling three elevation points (base, mid, and peak) per resort. The coordinates come from Open-Meteo's place search where it lists them. Otherwise, the resort's published coordinate is used for all three points.

In [2]:
RESORTS = {
    "Whistler Blackcomb": {"base": (50.12, -122.95, 675), "mid": (50.11, -122.95, 1480), "high":(50.06, -122.96, 2284)},
    "Big White": {"base": (49.72, -118.93, 1524), "mid": (49.72, -118.92, 1905), "high":(49.75, -118.94, 2286)},
    "Sun Peaks": {"base": (50.88, -119.91, 1186), "mid": (50.91, -119.91, 1634), "high":(50.94, -119.91, 2081)},
    "Revelstoke": {"base": (50.96, -118.16, 512), "mid": (50.96, -118.16, 1368), "high":(50.96, -118.16, 2225)},
    "Lake Louise": {"base": (51.46, -116.09, 1646), "mid": (51.46, -116.09, 2141), "high":(51.46, -116.09, 2636)},
    "Mont Tremblant": {"base": (46.21, -74.58, 230), "mid": (46.23, -74.58, 552), "high":(46.25, -74.58, 875)},
    "Mont Sainte-Anne": {"base": (47.07, -70.91, 175), "mid": (47.07, -70.91, 488), "high":(47.07, -70.91, 800)},
}

REGION = {"Whistler Blackcomb": "west", "Big White": "west", "Sun Peaks": "west", 
          "Revelstoke": "west", "Lake Louise": "west", "Mont Tremblant": "east", "Mont Sainte-Anne": "east"}
POINTS = ["base", "mid", "high"]

START_DATE = "2005-01-01"
END_DATE = "2026-05-31"

DAILY_VARS = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "apparent_temperature_mean", "apparent_temperature_min", "apparent_temperature_max",
    "wet_bulb_temperature_2m_mean", "wet_bulb_temperature_2m_min", "wet_bulb_temperature_2m_max",
    "dew_point_2m_mean", "relative_humidity_2m_mean", "precipitation_sum",
    "precipitation_hours", "rain_sum", "snowfall_sum", "snowfall_water_equivalent_sum",
    "sunshine_duration", "daylight_duration", "cloud_cover_mean", "shortwave_radiation_sum",
    "wind_speed_10m_max", "wind_gusts_10m_max", "surface_pressure_mean", 
    "soil_temperature_0_to_7cm_mean", "weather_code",
]
HOURLY_VARS = [
    "snowfall", "snow_depth", "temperature_2m", "wet_bulb_temperature_2m",
    "relative_humidity_2m", "dew_point_2m", "shortwave_radiation", "wind_gusts_10m",
    "wind_speed_10m", "soil_temperature_0_to_7cm", "cloud_cover", "weather_code",
    "surface_pressure", "precipitation", "rain", "apparent_temperature",
]
MODEL = "best_match"

In [3]:
HEADERS = {"User-Agent": "AC209A-SnowPredict (Harvard AC209A Fall 2026 project)"}
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
RAW_DIR = "data/raw"

os.makedirs(RAW_DIR, exist_ok=True)

def slug(name):
    return name.replace(" ", "_").replace("-", "_").replace(".", "")

def raw_path(name, point, year):
    return f"{RAW_DIR}/{slug(name)}_{point}_{year}.json"

## 1.2 API call - guarded, validated, cached

One request per resort, point, and calendar year. The guard fetches only if the file is not already on disk. The validation resufes to cache a bad or partial response, so a cached file can be trusted. 429 and 5xx are retried with backoff and any other 4xx raises.

In [4]:
def validate_payload(j, start, end):
    """Check if the JSON is a complete archive response for [start, end]"""
    if "error" in j:
        raise ValueError(f"API error: {j.get("reason")}")
    for key in ("daily", "hourly"):
        if key not in j or "time" not in j[key]:
            raise ValueError(f"response has no '{key}' block")
    n_days = (pd.Timestamp(end)-pd.Timestamp(start)).days + 1
    if len(j["daily"]["time"]) != n_days:
        raise ValueError(F"expected {n_days} daily rows, got {len(j["daily"]["time"])}")
    if len(j["hourly"]["time"]) != 24 * n_days:
        raise ValueError(f"expected {24*n_days} hourly rows, got {len(j["hourly"]["time"])}")
    missing_d = [v for v in DAILY_VARS if v not in j["daily"]]
    missing_h = [v for v in HOURLY_VARS if v not in j["hourly"]]
    if missing_d or missing_h:
        raise ValueError(f"missing daily {missing_d} / hourly {missing_h}")

def fetch_year(name, point, lat, lon, elev, year, retries=6):
    path = raw_path(name, point, year)
    if os.path.isfile(path):
        return path, "cached"
    s = max(pd.Timestamp(f"{year}-01-01"), pd.Timestamp(START_DATE)).strftime("%Y-%m-%d")
    e = min(pd.Timestamp(f"{year}-12-31"), pd.Timestamp(END_DATE)).strftime("%Y-%m-%d")
    params = {"latitude": lat, "longitude": lon, "elevation": elev, "start_date": s, "end_date": e,
              "daily": ",".join(DAILY_VARS), "hourly":",".join(HOURLY_VARS), "models": MODEL, "timezone": "auto"}
    r = None
    for attempt in range(retries):
        r =requests.get(ARCHIVE_URL, params=params, headers=HEADERS, timeout=120)
        if r.status_code == 429:
            wait = 60*(attempt+1)
            print(f" 429 on {name}/{point}/{year}; waiting {wait}s")
            time.sleep(wait); continue
        if r.status_code in (500, 502, 503, 504):
            time.sleep(10 * (attempt+1)); continue
        r.raise_for_status()
        r.encoding = "UTF-8"
        try:
            j = r.json()
            validate_payload(j, s, e)
        except (ValueError, KeyError) as err:            # not JSON, or incomplete JSON
            last = f"bad body ({type(err).__name__})"
            print(f"{last} on {name}/{point}/{year}; retrying in {30*(attempt+1)}s")
            time.sleep(30 * (attempt + 1)); continue
        with open(path, "w") as f:
            f.write(r.text)
        return path, "fetched"
    print(f"  SKIPPED {name}/{point}/{year}: last status {r.status_code}")
    return path, "failed"

In [5]:
years = range(int(START_DATE[:4]), int(END_DATE[:4]) + 1)
n_total = len(RESORTS) * len(POINTS) * len(years)
manifest, consecutive_fail = [], 0
counts = {"fetched": 0, "cached": 0, "failed": 0}

for name, pts in RESORTS.items():
    for point in POINTS:
        lat, lon, elev = pts[point]
        for y in years:
            path, status = fetch_year(name, point, lat, lon, elev, y)
            counts[status] += 1
            ok = status != "failed"
            manifest.append({"resort": name, "point": point, "year": y, "path": path,
                             "status": status, "bytes": os.path.getsize(path) if status != "failed" else 0, 
                             "cached_on": datetime.date.fromtimestamp(os.path.getmtime(path)) if status != "failed" else None})
            if status == "fetched":
                consecutive_fail = 0
                time.sleep(2)
            elif status == "failed":
                consecutive_fail += 1
                if consecutive_fail >= 5:
                    print(" 5 failures in a row; pausing 10 minutes before continuing")
                    time.sleep(10 * 60); consecutive_fail = 0
        done = sum(counts.values())
        print(f"[{done:3d}/{n_total}] {name}/{point}: fetched {counts["fetched"]}, cached {counts["cached"]}, failed {counts["failed"]}")

manifest = pd.DataFrame(manifest)
manifest.to_csv("data/raw_manifest.csv", index=False)
print()
print(f"Done. {counts["fetched"]} fetched this run, {counts["cached"]} already cached, {counts["failed"]} failed and still needed.")
print("Raw JSON on disk: %.1f MB across %d files" % (manifest["bytes"].sum() / 1e6, (manifest["status"] != "failed").sum()))
if (manifest["status"] == "failed").any():
    print("Re-run this cell later to fetch the failed files:")
    display(manifest.loc[manifest["status"] == "failed", ["resort", "point", "year"]])
manifest.head()

[ 22/462] Whistler Blackcomb/base: fetched 0, cached 22, failed 0
[ 44/462] Whistler Blackcomb/mid: fetched 0, cached 44, failed 0
[ 66/462] Whistler Blackcomb/high: fetched 0, cached 66, failed 0
[ 88/462] Big White/base: fetched 0, cached 88, failed 0
[110/462] Big White/mid: fetched 0, cached 110, failed 0
[132/462] Big White/high: fetched 0, cached 132, failed 0
[154/462] Sun Peaks/base: fetched 0, cached 154, failed 0
[176/462] Sun Peaks/mid: fetched 0, cached 176, failed 0
[198/462] Sun Peaks/high: fetched 0, cached 198, failed 0
[220/462] Revelstoke/base: fetched 0, cached 220, failed 0
[242/462] Revelstoke/mid: fetched 0, cached 242, failed 0
[264/462] Revelstoke/high: fetched 0, cached 264, failed 0
[286/462] Lake Louise/base: fetched 0, cached 286, failed 0
[308/462] Lake Louise/mid: fetched 0, cached 308, failed 0
[330/462] Lake Louise/high: fetched 0, cached 330, failed 0
[352/462] Mont Tremblant/base: fetched 0, cached 352, failed 0
[374/462] Mont Tremblant/mid: fetched 0,

,resort,point,year,path,status,bytes,cached_on
0,Whistler Blackcomb,base,2005,data/raw/Whistler_Blackcomb_base_2005.json,cached,838986,2026-10-31
1,Whistler Blackcomb,base,2006,data/raw/Whistler_Blackcomb_base_2006.json,cached,840744,2026-10-31
2,Whistler Blackcomb,base,2007,data/raw/Whistler_Blackcomb_base_2007.json,cached,838811,2026-10-31
3,Whistler Blackcomb,base,2008,data/raw/Whistler_Blackcomb_base_2008.json,cached,843344,2026-10-31
4,Whistler Blackcomb,base,2009,data/raw/Whistler_Blackcomb_base_2009.json,cached,843356,2026-10-31


## 1.3 Parse into one table

Hourly data are first concatenated across years for each resort-point. Then, aggregated into three local-time dayparts:

| Daypart | Hours (local) | Attached to |
| ----- | ----- | ----- |
| `night` | 18:00 previous day - 07:59 | The following date (it sets that morning's surface) |
| `am` | 08:00 - 11:59 | same date |
| `pm` | 12:00 - 17:59 | same date |

The parse produce three tables, each derived from the previous one. So, a different daypart definition later needs no re-parsing of the JSON.

| File | One row = | Purpose |
| ----- | ----- | ----- |
| `data/weather_hourly_long.parquet` | resort, point, hour (+ daypart label) | hourly aggregate, re-aggregate later |
| `data/weather_daypart_long.parquet` | resort, point, date, daypart | daypart-level |
| `data/weather_daily_wide.csv` | resort, point, date | daily block + daypart summaries as columns |

Each daypart summary is merged onto the daily block with `validate="one-to-one"` and a row-count assert.

In [6]:
DAYPART_AGG = {
    "temp_mean":       ("temperature_2m", "mean"),
    "temp_min":        ("temperature_2m", "min"),
    "temp_max":        ("temperature_2m", "max"),
    "hours_above_0c":  ("temperature_2m", lambda s: (s > 0).sum()),
    "feels_mean":      ("apparent_temperature", "mean"),
    "wetbulb_mean":    ("wet_bulb_temperature_2m", "mean"),
    "wetbulb_max":     ("wet_bulb_temperature_2m", "max"),
    "rh_mean":         ("relative_humidity_2m", "mean"),
    "dewpoint_mean":   ("dew_point_2m", "mean"),
    "radiation_mean":  ("shortwave_radiation", "mean"),
    "wind_mean":       ("wind_speed_10m", "mean"),
    "gust_max":        ("wind_gusts_10m", "max"),
    "soil_temp_mean":  ("soil_temperature_0_to_7cm", "mean"),
    "cloud_mean":      ("cloud_cover", "mean"),
    "weather_code_max":("weather_code", "max"),            # most severe code in the block
    "pressure_mean":   ("surface_pressure", "mean"),
    "precip_sum":      ("precipitation", "sum"),
    "rain_sum":        ("rain", "sum"),
    "snowfall_sum":    ("snowfall", "sum"),
    "snow_depth_mean": ("snow_depth", "mean"),
    "snow_depth_max":  ("snow_depth", "max"),
    "n_hours":         ("temperature_2m", "size"),         # hours in the block (6/4/14 expected)
}

In [7]:
def load_point(name, point):
    """All years of one resort-point -> dailly dataframe, hourly dataframe, api elevation"""
    paths = manifest.loc[(manifest["resort"] == name) & (manifest["point"] == point), "path"]
    dailies, hourlies, elev = [], [], None
    for p in sorted(paths):
        with open(p) as f:
            j = json.load(f)
        dailies.append(pd.DataFrame(j["daily"]))
        hourlies.append(pd.DataFrame(j["hourly"]))
        elev = j.get("elevation", elev)
    daily = pd.concat(dailies, ignore_index=True).rename(columns={"time": "date"})
    daily["date"] = pd.to_datetime(daily["date"])
    hourly = pd.concat(hourlies, ignore_index=True)
    hourly["time"] = pd.to_datetime(hourly["time"])
    return daily, hourly, elev

In [8]:
def label_dayparts(hourly):
    # daypart label and the ski-day date each hour belongs to (18:00-23:59 -> next day's night)
    hour = hourly["time"].dt.hour
    return hourly.assign(
        daypart = np.select([hour < 8, hour < 12, hour < 18], ["night", "am", "pm"], default="night"),
        date = hourly["time"].dt.normalize() + pd.to_timedelta((hour >= 18).astype(int), unit="D"),
    )

def daypart_long(hourly):
    # one row per (date, daypart) with the summaries in DAYPART_AGG
    return hourly.groupby(["date", "daypart"]).agg(**DAYPART_AGG).reset_index()

def daypart_wide(part):
    #pivot the long daypart table to one row per date with _night/_am/_pm columns
    wide = part.set_index(["date", "daypart"]).unstack("daypart")
    wide.columns = [f"{v}_{p}" for v, p in wide.columns]
    return wide.reset_index()

In [9]:
daily_frames, hourly_frames, part_frames = [], [], []
for name, pts in RESORTS.items():
    for point in POINTS:
        daily, hourly, elev = load_point(name, point)
        hourly = label_dayparts(hourly)
        part = daypart_long(hourly)
        n_before = len(daily)
        df = daily.merge(daypart_wide(part), on="date", how="left", validate="one_to_one")
        assert len(df) == n_before, f"{name}/{point}: merge changed the row count"
        for frame in (df, hourly, part):
            frame.insert(0, "point", point)
            frame.insert(0, "resort", name)
        df["region"] = REGION[name]
        df["elevation_requested_m"] = pts[point][2]
        df["elevation_api_m"] = elev
        daily_frames.append(df)
        hourly_frames.append(hourly)
        part_frames.append(part)

In [10]:
def tidy(frames, keys):
    out = pd.concat(frames, ignore_index=True).sort_values(keys).reset_index(drop=True)
    assert not out.duplicated(keys).any(), f"duplicate rows on {keys}"
    return out

In [11]:
hourly_long = tidy(hourly_frames, ["resort", "point", "time"])
daypart_tbl = tidy(part_frames, ["resort", "point", "date", "daypart"])
raw = tidy(daily_frames, ["resort", "point", "date"])

In [12]:
for tbl in (hourly_long, daypart_tbl, raw):
    for c in [c for c in tbl.columns if c.startswith("snow_depth")]:
        tbl[c] = tbl[c] * 100
raw["sunshine_hours"] = raw.pop("sunshine_duration") / 3600
raw["daylight_hours"] = raw.pop("daylight_duration") / 3600

In [13]:
hourly_long.to_parquet("data/weather_hourly_long.parquet", index=False)
daypart_tbl.to_parquet("data/weather_daypart_long.parquet", index=False)
raw.to_csv("data/weather_daily_wide.csv", index=False)

for label, tbl, path in [("hourly long", hourly_long, "data/weather_hourly_long.parquet"),
                         ("daypart long", daypart_tbl, "data/weather_hourly_long.parquet"),
                         ("daily wide", raw, "data/weather_daily_wide.csv")]:
    print(f"{label:13s} {str(tbl.shape):18s} -> {os.path.getsize(path)/1e6:6.1f} MB")
raw.head()

hourly long   (3941784, 21)      ->   80.1 MB
daypart long  (492744, 26)       ->   80.1 MB
daily wide    (164241, 97)       ->  123.7 MB


,resort,point,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,apparent_temperature_min,apparent_temperature_max,wet_bulb_temperature_2m_mean,wet_bulb_temperature_2m_min,wet_bulb_temperature_2m_max,dew_point_2m_mean,relative_humidity_2m_mean,precipitation_sum,precipitation_hours,rain_sum,snowfall_sum,snowfall_water_equivalent_sum,cloud_cover_mean,...,rain_sum_am,rain_sum_night,rain_sum_pm,snowfall_sum_am,snowfall_sum_night,snowfall_sum_pm,snow_depth_mean_am,snow_depth_mean_night,snow_depth_mean_pm,snow_depth_max_am,snow_depth_max_night,snow_depth_max_pm,n_hours_am,n_hours_night,n_hours_pm,region,elevation_requested_m,elevation_api_m,sunshine_hours,daylight_hours
0,Big White,base,2005-01-01,-10.1,-7.5,-14.8,-14.5,-18.1,-12.2,-11.2,-14.6,-9.4,-12.5,83,0.0,0.0,0.0,0.14,0.2,86,...,0.0,0.0,0.0,0.14,0.0,0.0,66.0,66.000000,65.666667,66.0,66.0,66.0,4.0,8.0,6.0,west,1524,1524.0,5.78755,8.207600
1,Big White,base,2005-01-02,-14.3,-12.8,-15.8,-18.9,-20.2,-17.2,-15.6,-16.7,-14.4,-18.4,72,0.0,0.0,0.0,0.00,0.0,30,...,0.0,0.0,0.0,0.00,0.0,0.0,65.0,65.000000,65.000000,65.0,65.0,65.0,4.0,14.0,6.0,west,1524,1524.0,8.00000,8.227019
2,Big White,base,2005-01-03,-15.5,-11.1,-18.1,-20.0,-22.4,-15.7,-16.8,-18.8,-13.1,-20.1,68,0.0,0.0,0.0,0.00,0.0,18,...,0.0,0.0,0.0,0.00,0.0,0.0,64.0,64.357143,64.000000,64.0,65.0,64.0,4.0,14.0,6.0,west,1524,1524.0,8.00000,8.247897
3,Big White,base,2005-01-04,-16.7,-13.5,-19.3,-21.4,-24.1,-18.2,-17.8,-20.0,-15.2,-20.8,71,0.0,0.0,0.0,0.00,0.0,16,...,0.0,0.0,0.0,0.00,0.0,0.0,64.0,64.000000,64.000000,64.0,64.0,64.0,4.0,14.0,6.0,west,1524,1524.0,8.00000,8.270189
4,Big White,base,2005-01-05,-13.5,-7.4,-17.5,-18.0,-22.2,-11.5,-15.0,-18.7,-9.8,-24.2,43,0.0,0.0,0.0,0.00,0.0,25,...,0.0,0.0,0.0,0.00,0.0,0.0,63.0,63.571429,63.000000,63.0,64.0,63.0,4.0,14.0,6.0,west,1524,1524.0,8.00000,8.293858


# 2. Understanding the data

**One row** = one resort, one elevation point, one calendar day (local time). Columns come in three groups.

**Identifiers:** `resort` (nominal, 7 levels), `point` (ordinal: base < mid < high), `date`, `region` (west/east), `elevation_requested_m` (what we put in), `elevation_api_m` (what Open-Meteo used).

**Daily summaries** (from the API's daily block):

| Column | Meaning | Unit |
| ----- | ----- | ----- |
| `temperature_2m_mean/max/min` | air temperature at 2m, adjusted to requested elevation | °C |
| `apparent_temperature_mean/min/max` | feels-like temperature (wind chill, humidity) | °C |
| `wet_bulb_temperature_2m_mean/min/max` | wet-bulb temperature; rain-vs-snow line sits near 0-1 °C | °C |
| `dew_point_2m_mean`, `relative_humidity_2m_mean` | air moisture: dry powder vs heavy wet snow | °C |
| `precipitation_sum`, `precipitation_hours`, `rain_sum` | total precipitation (water equivalent), hours with any, liquid part | mm, h, mm |
| `snowfall_sum`, `snowfall_water_equivalent_sum` | new snow depth and its water content (ratio = density) | cm, mm |
| `sunshine_hours`, `daylight_hours`, `cloud_cover_mean` | hours of direct sun (WMO > 120 W/m^2), day length, mean cloud | h, h, % |
| `shortwave_radiation_sum` | incoming solar energy (sun crust, melt) | MJ/m^2 |
| `wind_speed_10m_max`, `wind_gusts_10m_max` | daily peak wind and gust at 10 m | km/h |
| `surface_pressure_mean` | mean surface pressure (day-to-day change = fronts) | hPa |
| `soil_temperature_0_to_7cm_mean` | top-soil temperature (frozen ground holds base) | °C |
| `weather_code` | WMO code of the day's most severe weather (nominal) | code |

**Daypart summaries** (from the hourly block), each suffixed `_night`, `_am`, `_pm`: `temp_mean/min/max`, `hours_above_0c`, `feels_mean`, `wetbulb_mean/max`, `rh_mean`, `dewpoint_mean`, `radiation_mean`, `wind_mean`, `gust_max`, `soil_temp_mean`, `cloud_mean`, `weather_code_max`, `pressure_mean`, `precip_sum`, `rain_sum`, `snowfall_sum`, `snow_depth_mean/max` (cm), `n_hours`. Snow depth exists only hourly, so the daypart columns are its only source.

In [14]:
# TODO:


In [15]:
# TODO:

In [16]:
# TODO:

# 3. Cleaning and preprocessing

Rule: do not fill a missing value by default. Measure first, look for pattern, then decide and say what was dropped.

## 3.1 Missing data
`isna().mean()` is a proportion (mean of a boolean). A rate that varies by resort, point, or month is evidence against MCAR. It cannot by itself separate MAR from MNAR. With `best_match`, a variable that only one source model provides may be missing for whole periods.

In [17]:
# TODO:

## 3.2 Unusual or impossible values

Each flag is either an error (fix or drop) or a real observation (keep). The flagged rows are shown so the decision is based on evidence.

In [18]:
# TODO:

# 3.3 Ski-season table and season identifiers

Modeling rows: 1 December - 30 April. The raw table keeps all days so that lagged features for early-December dates (which need November) are available. A season is labelled by the calendar year it ends (Dec 2010 - April 2011 → `season = 2011`).

In [19]:
# TODO:

## 3.4 Scale of the features

Reported so we remember that linear/regularized regression and k-NN need standardization inside the pipeline (only on the training fold). Tree models do not.

In [20]:
# TODO:

## 3.5 Information leakage

Task: on date t, using only rows with date <= t, predict conditions on date t+30 at the same resort and point.

- Features are functions of past rows only: lags, rolling means, day-of-season, resort/point/region indicators. Daypart columns of day t are legitimate features (the night before t is over by the time we predict). Daypart columns of day t+30 are targets.
- **Historical-average baseline**: for a target on day-of-season *d* at a resort-point, the mean of the target on day *d* across training seasons only. Computed inside the training folds, never on the full table. With dayparts, the baseline is daypart-specific in the same way. 
- **Split by season**, with test seasons later than training seasons. Every fitted step (scaler, imputer, baseline averages, mode) fits on training seasons only.
- A score from one split is not a result; validation seasons are for model selection, and the test seasons are scored once at the end.


In [21]:
# TODO:

## 3.6 Draft skiing-condition index (candidate target)

A transparent 0-100 index for the ski day (am + pm). Each component is clipped to [0, 1] and weighted. The weights are a **design choice** to be varied in Milestone 4. This is a calculated index, not observed skiing quality.

In [22]:
# TODO:

## 3.7 Plots

Most plots use the `mid` point. One shows the three points side by side.

In [23]:
# TODO: